# 03 – Process variants, rework and conformity

A *variant* is the exact sequence of activities of one order. Events are ordered by timestamp, then by row position in the file (timestamps have day precision only, so the order of events within the same day is uncertain).

In [1]:
import duckdb
con = duckdb.connect()
con.execute("""
CREATE OR REPLACE VIEW events_ord AS
SELECT *, file_row_number AS rn
FROM read_parquet('../data/bpi2019.parquet', file_row_number=true)
WHERE "case:concept:name" NOT IN (
  SELECT DISTINCT "case:concept:name"
  FROM read_parquet('../data/bpi2019.parquet')
  WHERE "time:timestamp" < TIMESTAMPTZ '2017-01-01')
""")
con.sql("""SELECT COUNT(*) AS events, COUNT(DISTINCT "case:concept:name") AS cases FROM events_ord""").df()

,events,cases
0,1593429,251654


## 1. Variants
One string per order: all its activities in order.

In [2]:
con.execute("""
CREATE OR REPLACE TABLE variants AS
SELECT "case:concept:name" AS case_id,
       string_agg("concept:name", ' > ' ORDER BY "time:timestamp", rn) AS variant
FROM events_ord GROUP BY 1
""")
con.sql("SELECT COUNT(DISTINCT variant) AS n_variants FROM variants").df()

,n_variants
0,11950


In [7]:
con.sql("""
SELECT variant, COUNT(*) AS cases,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM variants GROUP BY 1 ORDER BY cases DESC LIMIT 10
""").df()

,variant,cases,pct
0,Create Purchase Order Item > Vendor creates in...,50286,19.98
1,Create Purchase Order Item > Record Goods Rece...,30798,12.24
2,Create Purchase Order Item > Record Goods Receipt,12214,4.85
3,Create Purchase Order Item > Vendor creates in...,11383,4.52
4,Create Purchase Order Item > Receive Order Con...,9694,3.85
5,Create Purchase Requisition Item > Create Purc...,8921,3.54
6,Create Purchase Order Item > Vendor creates in...,8835,3.51
7,Create Purchase Order Item > Record Goods Rece...,7985,3.17
8,Create Purchase Order Item > Delete Purchase O...,5298,2.11
9,Create Purchase Order Item > Receive Order Con...,4244,1.69


In [10]:
for v, n, p in con.sql("""
SELECT variant, COUNT(*) AS cases,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM variants GROUP BY 1 ORDER BY cases DESC LIMIT 5
""").fetchall():
    print(f"{p}%  ({n} cases)\n   {v}\n")

19.98%  (50286 cas)
   Create Purchase Order Item > Vendor creates invoice > Record Goods Receipt > Record Invoice Receipt > Clear Invoice

12.24%  (30798 cas)
   Create Purchase Order Item > Record Goods Receipt > Vendor creates invoice > Record Invoice Receipt > Clear Invoice

4.85%  (12214 cas)
   Create Purchase Order Item > Record Goods Receipt

4.52%  (11383 cas)
   Create Purchase Order Item > Vendor creates invoice > Record Goods Receipt > Record Invoice Receipt > Remove Payment Block > Clear Invoice

3.85%  (9694 cas)
   Create Purchase Order Item > Receive Order Confirmation > Record Goods Receipt > Vendor creates invoice > Record Invoice Receipt > Clear Invoice



## 2. Rework
Rework = price or quantity change, item deletion, invoice cancellation or approval change.

In [3]:
con.sql("""
SELECT "concept:name" AS activity,
       COUNT(DISTINCT "case:concept:name") AS cases,
       ROUND(100.0 * COUNT(DISTINCT "case:concept:name")
             / (SELECT COUNT(DISTINCT "case:concept:name") FROM events_ord), 2) AS pct_of_cases
FROM events_ord
WHERE "concept:name" IN ('Change Price','Change Quantity','Cancel Invoice Receipt',
                         'Delete Purchase Order Item','Change Approval for Purchase Order')
GROUP BY 1 ORDER BY cases DESC
""").df()

,activity,cases,pct_of_cases
0,Change Quantity,17584,6.99
1,Change Price,11209,4.45
2,Delete Purchase Order Item,8839,3.51
3,Cancel Invoice Receipt,6448,2.56
4,Change Approval for Purchase Order,4377,1.74


In [9]:
con.sql("""
WITH r AS (
  SELECT "case:concept:name" AS c,
    MAX(CASE WHEN "concept:name" IN ('Change Price','Change Quantity','Cancel Invoice Receipt',
        'Delete Purchase Order Item','Change Approval for Purchase Order') THEN 1 ELSE 0 END) AS has_rework
  FROM events_ord GROUP BY 1)
SELECT SUM(has_rework) AS cases_with_rework, COUNT(*) AS cases,
       ROUND(100.0 * SUM(has_rework) / COUNT(*), 1) AS pct
FROM r
""").df()

,cases_with_rework,cases,pct
0,42891.0,251654,17.0


### Activities repeated within the same order

In [4]:
con.sql("""
WITH r AS (
  SELECT "case:concept:name" AS c, "concept:name" AS a, COUNT(*) AS n
  FROM events_ord GROUP BY 1, 2)
SELECT a AS activity, COUNT(*) AS cases_with_repeat
FROM r WHERE n > 1 GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").df()

,activity,cases_with_repeat
0,Record Goods Receipt,12671
1,Record Invoice Receipt,9620
2,Clear Invoice,6808
3,Vendor creates invoice,5274
4,Record Service Entry Sheet,4146
5,Change Quantity,2472
6,Remove Payment Block,1108
7,Change Approval for Purchase Order,869
8,Change Price,868
9,Cancel Invoice Receipt,410


## 3. Conformity: is the invoice receipt recorded before the goods receipt?
For orders with goods-receipt-based invoice verification (`gr_based = True`), the invoice should only be recorded after the goods receipt. For the other orders an earlier invoice is allowed.

The test uses *Record Invoice Receipt* (not *Vendor creates invoice*, which is the vendor's own step and can legitimately come first). Same-day cases have uncertain order.

In [11]:
con.sql("""
WITH f AS (
  SELECT "case:concept:name" AS c,
    ANY_VALUE("case:GR-Based Inv. Verif.") AS gr_based,
    MIN(CAST("time:timestamp" AS DATE)) FILTER (WHERE "concept:name"='Record Invoice Receipt') AS inv_rec,
    MIN(CAST("time:timestamp" AS DATE)) FILTER (WHERE "concept:name"='Record Goods Receipt') AS gr
  FROM events_ord GROUP BY 1)
SELECT gr_based,
  COUNT(*) FILTER (WHERE inv_rec IS NOT NULL AND gr IS NOT NULL) AS with_both,
  COUNT(*) FILTER (WHERE inv_rec < gr)  AS inv_rec_before_gr,
  COUNT(*) FILTER (WHERE inv_rec = gr)  AS same_day,
  COUNT(*) FILTER (WHERE inv_rec > gr)  AS gr_before_inv_rec
FROM f GROUP BY 1
""").df()

,gr_based,with_both,inv_rec_before_gr,same_day,gr_before_inv_rec
0,False,199180,12301,11446,175433
1,True,11110,0,272,10838
